# 载波频率偏移（CFO）估计与补偿

## 学习目标

- 理解载波频率偏移的成因及其对通信系统的影响
- 掌握分数频偏估计的导频辅助方法
- 掌握整数频偏估计技术
- 理解数据辅助CFO估计的判决反馈机制
- 掌握CFO补偿的时域和频域方法
- 通过Python实现深入理解CFO估计与补偿算法

## 1. CFO基础概念

### 1.1 载波频率偏移的成因

载波频率偏移（CFO）是指接收机的本地振荡器频率与发射机的载波频率之间存在的偏差。这种偏差主要由以下因素引起：

1. **晶体振荡器精度**：低成本晶体振荡器存在固有频率误差
2. **多普勒效应**：移动终端与基站之间的相对运动导致频率偏移
3. **温度漂移**：环境温度变化引起振荡器频率漂移
4. **载波同步误差**：收发双方载波同步不完美

### 1.2 CFO的数学模型

假设发射信号为 x(t)，载波频率为 f_c。由于CFO Δf 的存在，接收信号变为：

$$r(t) = x(t) \cdot e^{j2\pi \Delta f t} + n(t)$$

其中 n(t) 是加性高斯白噪声。

在离散域，采样后的信号表示为：

$$r[n] = x[n] \cdot e^{j2\pi \frac{\Delta f}{f_s} n} + n[n]$$

其中 f_s 是采样频率，ε = Δf / f_s 是归一化频偏。

### 1.3 CFO对OFDM系统的影响

CFO对OFDM系统会造成严重危害：

- **子载波间干扰（ICI）**：破坏子载波正交性
- **星座旋转**：每个子载波上的数据发生相位旋转
- **信噪比损失**：干扰导致有效SNR下降
- **误码率上升**：系统性能严重恶化

CFO大小分类：
- **分数频偏** |ε| < 0.5：主要引起相位旋转
- **整数频偏** |ε| >= 0.5：引起子载波移位，导致突发错误

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
f = np.linspace(-0.5, 0.5, 1000)
ax1.plot(f, np.exp(-f**2/0.02), "b-", linewidth=2, label="理想载波")
ax1.plot(f, np.exp(-(f-0.15)**2/0.02), "r-", linewidth=2, label="CFO偏移后")
ax1.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax1.set_xlabel("归一化频率", fontsize=12)
ax1.set_ylabel("功率谱密度", fontsize=12)
ax1.set_title("CFO对载波频率的影响", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
subcarriers = np.arange(-32, 32)
ax2.bar(subcarriers, np.exp(-subcarriers**2/100), width=0.8, alpha=0.7, label="理想正交")
ax2.bar(subcarriers + 3, np.exp(-(subcarriers-3)**2/100)*0.5, width=0.8, alpha=0.5, label="CFO导致ICI")
ax2.set_xlabel("子载波索引", fontsize=12)
ax2.set_ylabel("幅度", fontsize=12)
ax2.set_title("CFO导致的子载波间干扰(ICI)", fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 分数频偏估计（导频辅助方法）

### 2.1 导频辅助方法原理

导频辅助CFO估计利用已知位置的导频信号来估计频率偏移。通过比较接收导频与理想导频的相位差，可以计算出CFO。

对于两个连续的导频符号，其相位差为：

$$\Delta \phi = 2\pi \epsilon \cdot N_{sym}$$

其中 N_sym 是两个导频之间的符号数。

因此归一化频偏估计为：

$$\hat{\epsilon} = \frac{\Delta \phi}{2\pi N_{sym}}$$

这种方法的精度取决于导频的数量和信噪比。

In [ ]:
def pilot_assisted_cfo_estimation(rx_signal, pilot_pattern, n_fft, n_sym):
    """
    导频辅助CFO估计
    
    参数:
        rx_signal: 接收信号（时域）
        pilot_pattern: 导频位置索引
        n_fft: FFT大小
        n_sym: 符号数
    返回:
        epsilon: 归一化频偏估计值
    """
    n_pilots = len(pilot_pattern)
    phase_diff_sum = 0
    
    for i in range(len(pilot_pattern) - 1):
        pilot_idx = pilot_pattern[i]
        next_pilot_idx = pilot_pattern[i + 1]
        
        rx_fft_current = np.fft.fft(rx_signal[pilot_idx], n_fft)
        rx_fft_next = np.fft.fft(rx_signal[next_pilot_idx], n_fft)
        
        for k in range(n_pilots):
            if k < len(pilot_pattern):
                phase_diff = np.angle(rx_fft_next[pilot_pattern[k]]) - np.angle(rx_fft_current[pilot_pattern[k]])
                phase_diff_sum += phase_diff
    
    avg_phase_diff = phase_diff_sum / (len(pilot_pattern) * (len(pilot_pattern) - 1))
    epsilon = avg_phase_diff / (2 * np.pi)
    
    return epsilon


def moose_cfo_estimation(rx_pilot1, rx_pilot2, n_fft):
    """
    Moose方法：基于两个重复训练符号的CFO估计
    
    该方法利用两个相同的训练符号，通过它们之间的相位差估计CFO
    
    参数:
        rx_pilot1: 第一个训练符号（时域）
        rx_pilot2: 第二个训练符号（时域）
        n_fft: FFT大小
    返回:
        epsilon: 归一化频偏估计值
    """
    correlation = np.sum(rx_pilot1 * np.conj(rx_pilot2))
    phi = np.angle(correlation)
    epsilon = phi / (2 * np.pi)
    
    return epsilon


print("导频辅助CFO估计方法已定义")

In [ ]:
# 演示Moose方法的CFO估计
np.random.seed(42)

n_fft = 64
cfo_true = 0.15
snr_db = 20

pilot_symbol = np.random.randn(n_fft) + 1j * np.random.randn(n_fft)

n_samples = n_fft
n = np.arange(n_samples)
pilot_with_cfo = pilot_symbol * np.exp(1j * 2 * np.pi * cfo_true * n / n_fft)

signal_power = np.mean(np.abs(pilot_with_cfo)**2)
noise_power = signal_power / (10**(snr_db/10))
noise = np.sqrt(noise_power/2) * (np.random.randn(n_fft) + 1j * np.random.randn(n_fft))
rx_pilot = pilot_with_cfo + noise

rx_pilot1 = rx_pilot
rx_pilot2 = pilot_symbol * np.exp(1j * 2 * np.pi * cfo_true * (n + n_fft) / n_fft) + noise

cfo_estimated = moose_cfo_estimation(rx_pilot1, rx_pilot2, n_fft)

print(f"真实的归一化CFO: {cfo_true:.4f}")
print(f"估计的归一化CFO: {cfo_estimated:.4f}")
print(f"估计误差: {abs(cfo_true - cfo_estimated):.4f}")

### 2.2 分数频偏估计的Cramer-Rao下界(CRLB)

对于导频辅助方法，方差下界为：

$$\text{Var}(\hat{\epsilon}) \geq \frac{1}{4\pi^2 N \cdot \text{SNR}}$$

其中 N 是用于估计的样本数，SNR 是信噪比。

这说明：
- 样本数越多，估计方差越小
- 信噪比越高，估计越准确

In [ ]:
def crlb_cfo_estimation(n_samples, snr_db):
    """
    计算CFO估计的CRLB
    
    参数:
        n_samples: 样本数
        snr_db: 信噪比(dB)
    返回:
        crlb: 方差下界
    """
    snr = 10**(snr_db/10)
    crlb = 1 / (4 * np.pi**2 * n_samples * snr)
    return crlb


snr_range = np.arange(5, 35, 2)
n_samples_range = [64, 256, 1024]

fig, ax = plt.subplots(figsize=(10, 6))

for n_samples in n_samples_range:
    crlb_values = [crlb_cfo_estimation(n_samples, snr) for snr in snr_range]
    ax.semilogy(snr_range, crlb_values, "-o", label=f"N={n_samples}")

ax.set_xlabel("信噪比 (dB)", fontsize=12)
ax.set_ylabel("CRLB (方差)", fontsize=12)
ax.set_title("CFO估计的Cramer-Rao下界", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 整数频偏估计

### 3.1 整数频偏的问题

当 |ε| >= 0.5 时，导频辅助方法只能估计小数部分，因为相位旋转周期为 2π。整数部分需要额外的方法来估计。

整数频偏会导致：
- 子载波完全错位（移位）
- 接收数据完全错误
- 传统导频方法无法检测

### 3.2 基于相关性的整数频偏估计

一种常用的方法是利用OFDM符号前后部分的循环前缀相关性来估计整数频偏。

基本原理：
1. 找到循环前缀与对应数据之间的最大相关峰
2. 相关峰的位置包含了整数频偏信息
3. 通过搜索不同的整数偏移，找到使相关最大的值

In [ ]:
def integer_cfo_estimation(rx_signal, n_fft, n_cp, search_range=10):
    """
    整数频偏估计（基于循环前缀相关）
    
    参数:
        rx_signal: 接收信号（时域）
        n_fft: FFT大小
        n_cp: 循环前缀长度
        search_range: 搜索范围
    返回:
        int_cfo: 整数频偏估计值
    """
    rx_fft = np.fft.fft(rx_signal[n_cp:n_cp+n_fft], n_fft)
    
    max_corr = 0
    int_cfo = 0
    
    for offset in range(-search_range, search_range + 1):
        permuted_idx = np.arange(n_fft) + offset
        permuted_idx = permuted_idx % n_fft
        corr = np.abs(np.sum(rx_fft[permuted_idx]))
        
        if corr > max_corr:
            max_corr = corr
            int_cfo = offset
    
    return int_cfo


def training_based_int_cfo_estimation(rx_fft, pilot_positions, ideal_pilots, search_range=5):
    """
    基于训练序列的整数频偏估计
    
    利用训练序列中已知位置的导频来估计整数频偏
    
    参数:
        rx_fft: 接收信号的FFT（频域）
        pilot_positions: 导频位置索引
        ideal_pilots: 理想的导频值
        search_range: 搜索范围
    返回:
        int_cfo: 整数频偏估计值
    """
    n_pilots = len(pilot_positions)
    max_corr = 0
    int_cfo = 0
    
    for offset in range(-search_range, search_range + 1):
        shifted_positions = (np.array(pilot_positions) + offset) % len(rx_fft)
        
        rx_pilots = rx_fft[shifted_positions]
        corr = np.abs(np.sum(rx_pilots * np.conj(ideal_pilots)))
        
        if corr > max_corr:
            max_corr = corr
            int_cfo = offset
    
    return int_cfo


print("整数频偏估计方法已定义")

### 3.3 结合分数和整数频偏估计

在实际系统中，需要先估计分数频偏，补偿后再估计整数频偏：

1. **步骤1**：使用 Moose 或类似方法估计分数频偏
2. **步骤2**：补偿分数频偏
3. **步骤3**：使用整数频偏估计方法估计
4. **步骤4**：组合得到完整估计

In [ ]:
def combined_cfo_estimation(rx_signal, n_fft, n_cp, pilot_positions, ideal_pilots, search_range=5):
    """
    联合分数和整数频偏估计
    
    参数:
        rx_signal: 接收信号
        n_fft: FFT大小
        n_cp: 循环前缀长度
        pilot_positions: 导频位置
        ideal_pilots: 理想导频值
        search_range: 整数搜索范围
    返回:
        total_cfo: 总CFO（分数+整数）
    """
    rx_fft = np.fft.fft(rx_signal[n_cp:n_cp+n_fft], n_fft)
    
    rx_pilot_phases = np.angle(rx_fft[pilot_positions])
    ideal_phases = np.angle(ideal_pilots)
    phase_diff = rx_pilot_phases - ideal_phases
    
    frac_cfo = np.median(phase_diff) / (2 * np.pi)
    
    n_samples = len(rx_signal)
    n = np.arange(n_samples)
    compensated_signal = rx_signal * np.exp(-1j * 2 * np.pi * frac_cfo * n / n_fft)
    
    compensated_fft = np.fft.fft(compensated_signal[n_cp:n_cp+n_fft], n_fft)
    int_cfo = training_based_int_cfo_estimation(
        compensated_fft, pilot_positions, ideal_pilots, search_range
    )
    
    total_cfo = frac_cfo + int_cfo
    
    return total_cfo, frac_cfo, int_cfo


np.random.seed(123)

n_fft = 64
n_cp = 16
frac_cfo_true = 0.23
int_cfo_true = 2
total_cfo_true = frac_cfo_true + int_cfo_true

pilot_positions = np.array([0, 16, 32, 48])
ideal_pilots = np.array([1, 1, 1, 1], dtype=complex)

tx_signal = np.random.randn(n_fft + n_cp) + 1j * np.random.randn(n_fft + n_cp)
n = np.arange(len(tx_signal))
rx_signal = tx_signal * np.exp(1j * 2 * np.pi * total_cfo_true * n / n_fft)
rx_signal += 0.01 * (np.random.randn(len(tx_signal)) + 1j * np.random.randn(len(tx_signal)))

total_est, frac_est, int_est = combined_cfo_estimation(
    rx_signal, n_fft, n_cp, pilot_positions, ideal_pilots
)

print(f"真实CFO: {total_cfo_true:.4f} (分数: {frac_cfo_true:.4f}, 整数: {int_cfo_true})")
print(f"估计CFO: {total_est:.4f} (分数: {frac_est:.4f}, 整数: {int_est})")

## 4. 数据辅助CFO估计（判决反馈）

### 4.1 判决反馈原理

当没有导频可用时，可以利用判决反馈来进行CFO估计。基本思想是：

1. **初始估计**：使用训练序列进行初始CFO估计
2. **判决**：对接收数据进行解调和解码
3. **重新编码**：将判决结果重新编码为发射信号
4. **比较**：将重新编码的信号与接收信号比较，估计残留CFO
5. **更新**：不断迭代更新CFO估计

In [ ]:
def decision_feedback_cfo_estimation(rx_signal, n_fft, n_cp, initial_cfo, n_iter=5):
    """
    判决反馈CFO估计
    
    参数:
        rx_signal: 接收信号
        n_fft: FFT大小
        n_cp: 循环前缀长度
        initial_cfo: 初始CFO估计值
        n_iter: 迭代次数
    返回:
        cfo_estimate: 估计的CFO值
    """
    cfo = initial_cfo
    n_samples = len(rx_signal)
    n = np.arange(n_samples)
    
    for iteration in range(n_iter):
        compensated = rx_signal * np.exp(-1j * 2 * np.pi * cfo * n / n_fft)
        
        rx_fft = np.fft.fft(compensated[n_cp:n_cp+n_fft], n_fft)
        
        tx_reconstructed = np.sign(np.real(rx_fft)) + 1j * np.sign(np.imag(rx_fft))
        
        if iteration < n_iter - 1:
            phase_correction = 0
            for i in range(len(rx_fft) - 1):
                phase_diff = np.angle(rx_fft[i+1] * np.conj(rx_fft[i]))
                phase_correction += phase_diff
            
            avg_phase = phase_correction / (len(rx_fft) - 1)
            residual_cfo = avg_phase / (2 * np.pi)
            
            cfo = cfo + 0.5 * residual_cfo
    
    return cfo


print("判决反馈CFO估计方法已定义")

### 4.2 判决反馈的收敛性分析

判决反馈方法的收敛性取决于：

- **初始估计精度**：初始CFO越准确，收敛越快
- **信道条件**：低SNR下误判概率高，可能导致错误传播
- **迭代步长**：步长过大可能导致振荡，步长过小收敛慢
- **调制阶数**：高阶调制对CFO更敏感，需要更精确的估计

In [ ]:
def test_decision_feedback_convergence():
    np.random.seed(456)
    
    n_fft = 64
    n_cp = 16
    cfo_true = 0.35
    snr_db = 15
    
    n_symbols = 20
    tx_data = np.random.choice([1+1j, 1-1j, -1+1j, -1-1j], size=(n_symbols, n_fft))
    
    rx_signals = []
    for i in range(n_symbols):
        symbol_with_cp = np.concatenate([tx_data[i, -n_cp:], tx_data[i]])
        n = np.arange(len(symbol_with_cp))
        rx = symbol_with_cp * np.exp(1j * 2 * np.pi * cfo_true * n / n_fft)
        rx_signals.append(rx)
    
    rx_signal = np.concatenate(rx_signals)
    
    signal_power = np.mean(np.abs(rx_signal)**2)
    noise_power = signal_power / (10**(snr_db/10))
    rx_signal += np.sqrt(noise_power/2) * (np.random.randn(len(rx_signal)) + 1j * np.random.randn(len(rx_signal)))
    
    initial_cfo = 0.2
    cfo_estimated = decision_feedback_cfo_estimation(rx_signal, n_fft, n_cp, initial_cfo, n_iter=10)
    
    print(f"真实CFO: {cfo_true:.4f}")
    print(f"初始估计: {initial_cfo:.4f}")
    print(f"判决反馈估计: {cfo_estimated:.4f}")
    print(f"估计误差: {abs(cfo_true - cfo_estimated):.4f}")


test_decision_feedback_convergence()

## 5. CFO补偿方法

### 5.1 时域补偿

时域补偿是最直接的CFO补偿方法。通过将接收信号乘以一个相位旋转因子来抵消CFO的影响：

$$r_{compensated}[n] = r[n] \cdot e^{-j2\pi \hat{\epsilon} n / N_{FFT}}$$

优点：
- 实现简单
- 在FFT之前完成，不引入额外复杂度

缺点：
- 需要精确的CFO估计
- 噪声会随着补偿过程放大

In [ ]:
def time_domain_cfo_compensation(rx_signal, cfo_estimate, n_fft):
    """
    时域CFO补偿
    
    参数:
        rx_signal: 接收信号（时域）
        cfo_estimate: CFO估计值（归一化）
        n_fft: FFT大小
    返回:
        compensated: CFO补偿后的信号
    """
    n_samples = len(rx_signal)
    n = np.arange(n_samples)
    
    compensation_factor = np.exp(-1j * 2 * np.pi * cfo_estimate * n / n_fft)
    compensated = rx_signal * compensation_factor
    
    return compensated


def frequency_domain_cfo_compensation(rx_fft, cfo_estimate):
    """
    频域CFO补偿
    
    参数:
        rx_fft: 接收信号的FFT
        cfo_estimate: CFO估计值
    返回:
        compensated: CFO补偿后的FFT信号
    """
    n_subcarriers = len(rx_fft)
    k = np.arange(n_subcarriers)
    
    phase_compensation = np.exp(-1j * 2 * np.pi * cfo_estimate * k / n_subcarriers)
    
    compensated = rx_fft * phase_compensation
    
    return compensated


print("CFO补偿方法已定义")

### 5.2 频域补偿

频域补偿在FFT之后进行，主要补偿CFO导致的相位旋转：

$$Y_{compensated}[k] = Y[k] \cdot e^{-j2\pi \hat{\epsilon} k / N}$$

其中 k 是子载波索引，N 是FFT大小。

优点：
- 可以针对每个子载波单独补偿
- 与信道估计结合方便

缺点：
- 不能消除ICI（子载波间干扰）
- 需要精确的CFO估计

### 5.3 ICI自消除技术

对于残留的ICI，可以使用ICI自消除技术。基本原理是在发射端对相邻的两个符号使用相反的加权，在接收端通过合并来消除ICI。

发射信号设计：
$$X'[k] = \alpha_k X[k] + \beta_k X[k+1]$$

接收端合并：
$$Y_{cancel}[k] = Y'[k] + \gamma Y'[k+1]$$

通过精心设计系数，可以显著降低ICI。

In [ ]:
def demo_cfo_compensation():
    np.random.seed(789)
    
    n_fft = 64
    n_cp = 16
    n_symbols = 5
    cfo_true = 0.35
    snr_db = 20
    
    tx_constellation = np.array([1+1j, 1-1j, -1+1j, -1-1j]) / np.sqrt(2)
    tx_data = np.random.choice(tx_constellation, size=(n_symbols, n_fft))
    
    rx_signals = []
    for i in range(n_symbols):
        symbol_with_cp = np.concatenate([tx_data[i, -n_cp:], tx_data[i]])
        n = np.arange(len(symbol_with_cp))
        rx = symbol_with_cp * np.exp(1j * 2 * np.pi * cfo_true * n / n_fft)
        rx_signals.append(rx)
    
    rx_signal = np.concatenate(rx_signals)
    signal_power = np.mean(np.abs(rx_signal)**2)
    noise_power = signal_power / (10**(snr_db/10))
    rx_signal += np.sqrt(noise_power/2) * (np.random.randn(len(rx_signal)) + 1j * np.random.randn(len(rx_signal)))
    
    symbols = []
    for i in range(n_symbols):
        start = i * (n_fft + n_cp)
        symbols.append(rx_signal[start:start+n_fft+n_cp])
    
    compensated_signal = time_domain_cfo_compensation(rx_signal, cfo_true, n_fft)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    for i, ax in enumerate(axes.flat[:3]):
        if i < n_symbols:
            symbol = symbols[i]
            rx_fft_no_comp = np.fft.fft(symbol[n_cp:n_cp+n_fft], n_fft)
            ax.scatter(np.real(rx_fft_no_comp), np.imag(rx_fft_no_comp), alpha=0.5, s=20)
            ax.set_title(f"无补偿 - 符号 {i+1}", fontsize=11)
            ax.set_xlabel("实部")
            ax.set_ylabel("虚部")
            ax.grid(True, alpha=0.3)
            ax.set_xlim(-2, 2)
            ax.set_ylim(-2, 2)
    
    for i, ax in enumerate(axes.flat[3:6]):
        if i < n_symbols:
            comp_symbol = compensated_signal[i*(n_fft+n_cp):(i+1)*(n_fft+n_cp)]
            rx_fft_comp = np.fft.fft(comp_symbol[n_cp:n_cp+n_fft], n_fft)
            ax.scatter(np.real(rx_fft_comp), np.imag(rx_fft_comp), alpha=0.5, s=20)
            ax.set_title(f"补偿后 - 符号 {i+1}", fontsize=11)
            ax.set_xlabel("实部")
            ax.set_ylabel("虚部")
            ax.grid(True, alpha=0.3)
            ax.set_xlim(-2, 2)
            ax.set_ylim(-2, 2)
    
    plt.suptitle(f"CFO补偿效果演示 (CFO={cfo_true:.3f}, SNR={snr_db}dB)", fontsize=14)
    plt.tight_layout()
    plt.show()


demo_cfo_compensation()

## 6. Python实现：CFO估计和补偿演示

下面是一个完整的CFO估计和补偿系统演示，涵盖了实际通信中的各种场景。

In [ ]:
class OFDMCFOEstimator:
    """
    OFDM系统CFO估计与补偿完整类
    """
    
    def __init__(self, n_fft=64, n_cp=16, pilot_spacing=4):
        self.n_fft = n_fft
        self.n_cp = n_cp
        self.pilot_spacing = pilot_spacing
        
        self.pilot_positions = np.arange(0, n_fft, pilot_spacing)
        self.n_pilots = len(self.pilot_positions)
        
        self.pilot_values = np.random.choice([-1, 1], size=self.n_pilots)
    
    def generate_pilot_sequence(self):
        """生成导频序列""" 
        freq_pilot = np.zeros(self.n_fft, dtype=complex)
        freq_pilot[self.pilot_positions] = self.pilot_values
        
        time_symbol = np.fft.ifft(freq_pilot)
        symbol_with_cp = np.concatenate([time_symbol[-self.n_cp:], time_symbol])
        
        return symbol_with_cp, freq_pilot
    
    def estimate_fractional_cfo(self, rx_signal):
        """估计分数频偏（基于导频）""" 
        if len(rx_signal) > self.n_fft + self.n_cp:
            rx_signal = rx_signal[self.n_cp:self.n_cp+self.n_fft]
        
        rx_fft = np.fft.fft(rx_signal, self.n_fft)
        
        rx_pilot_phases = np.angle(rx_fft[self.pilot_positions])
        ideal_phases = np.angle(self.pilot_values)
        
        phase_diff = rx_pilot_phases - ideal_phases
        phase_diff_unwrapped = np.unwrap(phase_diff)
        avg_phase_diff = np.mean(phase_diff_unwrapped)
        
        frac_cfo = avg_phase_diff / (2 * np.pi)
        
        return frac_cfo
    
    def compensate_cfo(self, rx_signal, cfo_estimate):
        """时域CFO补偿""" 
        n = np.arange(len(rx_signal))
        compensation = np.exp(-1j * 2 * np.pi * cfo_estimate * n / self.n_fft)
        return rx_signal * compensation
    
    def estimate_integer_cfo(self, rx_fft):
        """估计整数频偏（基于相关）""" 
        max_corr = 0
        best_offset = 0
        search_range = 5
        
        for offset in range(-search_range, search_range + 1):
            shifted_pilot_positions = (self.pilot_positions + offset) % self.n_fft
            corr = np.abs(np.sum(rx_fft[shifted_pilot_positions] * np.conj(self.pilot_values)))
            if corr > max_corr:
                max_corr = corr
                best_offset = offset
        
        return best_offset
    
    def process(self, rx_signal, cfo_true=None):
        """完整处理流程""" 
        frac_cfo = self.estimate_fractional_cfo(rx_signal)
        
        compensated = self.compensate_cfo(rx_signal, frac_cfo)
        
        rx_fft = np.fft.fft(compensated[self.n_cp:self.n_cp+self.n_fft], self.n_fft)
        
        int_cfo = self.estimate_integer_cfo(rx_fft)
        
        total_cfo = frac_cfo + int_cfo
        
        return {
            "frac_cfo": frac_cfo,
            "int_cfo": int_cfo,
            "total_cfo": total_cfo,
            "compensated_signal": compensated,
            "fft_result": rx_fft
        }


def test_complete_system():
    np.random.seed(42)
    
    estimator = OFDMCFOEstimator(n_fft=64, n_cp=16, pilot_spacing=8)
    pilot_seq, freq_pilot = estimator.generate_pilot_sequence()
    
    frac_cfo_true = 0.27
    int_cfo_true = 2
    total_cfo_true = frac_cfo_true + int_cfo_true
    
    n = np.arange(len(pilot_seq))
    rx_pilot = pilot_seq * np.exp(1j * 2 * np.pi * total_cfo_true * n / estimator.n_fft)
    
    snr_db = 25
    signal_power = np.mean(np.abs(rx_pilot)**2)
    noise_power = signal_power / (10**(snr_db/10))
    rx_pilot += np.sqrt(noise_power/2) * (np.random.randn(len(rx_pilot)) + 1j * np.random.randn(len(rx_pilot)))
    
    result = estimator.process(rx_pilot)
    
    print("=" * 50)
    print("CFO估计与补偿系统测试结果")
    print("=" * 50)
    print(f"真实CFO: {total_cfo_true:.4f} (分数: {frac_cfo_true:.4f}, 整数: {int_cfo_true})")
    print(f"估计分数CFO: {result["frac_cfo"]:.4f}")
    print(f"估计整数CFO: {result["int_cfo"]}")
    print(f"总估计CFO: {result["total_cfo"]:.4f}")
    print(f"估计误差: {abs(total_cfo_true - result["total_cfo"]):.4f}")
    print("=" * 50)


test_complete_system()

In [ ]:
def cfo_estimation_performance():
    np.random.seed(999)
    
    n_fft = 64
    n_cp = 16
    n_trials = 100
    cfo_true = 0.35
    snr_range = [5, 10, 15, 20, 25, 30]
    
    estimator = OFDMCFOEstimator(n_fft=n_fft, n_cp=n_cp, pilot_spacing=8)
    pilot_seq, _ = estimator.generate_pilot_sequence()
    
    errors = []
    
    for snr_db in snr_range:
        trial_errors = []
        
        for trial in range(n_trials):
            n = np.arange(len(pilot_seq))
            rx_pilot = pilot_seq * np.exp(1j * 2 * np.pi * cfo_true * n / n_fft)
            
            signal_power = np.mean(np.abs(rx_pilot)**2)
            noise_power = signal_power / (10**(snr_db/10))
            rx_pilot += np.sqrt(noise_power/2) * (np.random.randn(len(rx_pilot)) + 1j * np.random.randn(len(rx_pilot)))
            
            frac_cfo = estimator.estimate_fractional_cfo(rx_pilot)
            
            error = abs(cfo_true - frac_cfo)
            trial_errors.append(error)
        
        errors.append(np.mean(trial_errors))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(snr_range, errors, "b-o", linewidth=2, markersize=8)
    ax.set_xlabel("信噪比 (dB)", fontsize=12)
    ax.set_ylabel("平均估计误差", fontsize=12)
    ax.set_title("CFO估计精度 vs SNR", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.set_yscale("log")
    plt.tight_layout()
    plt.show()


cfo_estimation_performance()

## 7. 思考题

1. **分析题**：如果CFO估计存在偏差，会对系统性能产生什么影响？请从ICI和相位旋转两个方面进行分析。

2. **设计题**：假设你需要设计一个高速移动通信系统的CFO跟踪算法，你会如何结合判决反馈和导频辅助方法？考虑移动场景下的多普勒频移。

3. **计算题**：对于一个子载波间隔为15kHz的5G NR系统，如果终端移动速度为300km/h，载波频率为3.5GHz，请计算最大多普勒频偏对应的归一化CFO（假设采样率为122.88MHz）。

4. **实现题**：如果让你实现一个基于EM（期望最大化）算法的联合CFO和信道估计器，你会如何设计？

5. **对比题**：比较 Moose 方法和基于导频的分数频偏估计方法的优缺点，讨论在不同场景下应该如何选择。

## 8. 总结

本notebook涵盖了载波频率偏移（CFO）估计与补偿的主要内容：

- **CFO基础**：理解了CFO的成因（晶振误差、多普勒效应、温度漂移）及其对OFDM系统的危害（ICI、相位旋转、性能下降）

- **分数频偏估计**：掌握了基于导频的估计方法和Moose方法，理解了CRLB下界对估计精度的限制

- **整数频偏估计**：理解了整数频偏与分数频偏的区别，掌握了基于相关性的整数频偏估计方法

- **数据辅助估计**：理解了判决反馈机制及其收敛性分析

- **CFO补偿**：掌握了时域补偿和频域补偿两种方法，了解了ICI自消除技术

- **实际实现**：通过Python代码实现了完整的CFO估计与补偿系统，并分析了不同SNR下的性能表现